# v5 Grid Search — Tìm SKIP_TOP_K tối ưu

Thử 4 cấu hình mining, so sánh để tìm sweet spot:

| Config | SKIP_TOP_K | Mining range | Nhận xét |
|--------|-----------|--------------|----------|
| A | 5 | rank 6-30 | Nhiều hard hơn |
| B | 8 | rank 9-30 | Hard-medium mix |
| C | 12 | rank 13-30 | Gần v5fix hiện tại |
| **D** | **14** | **rank 15-30** | **v5fix đang dùng** |

**Kết quả v5fix (D):** Recall@1=0.5418, Recall@5=0.7245, MRR=0.6307  
**Mục tiêu:** Tìm config vừa giữ R@1 cao vừa kéo R@5 lên gần 0.75+

## Cell 0 — Config & Imports

In [ ]:
import json, csv, time, random, gc
import numpy as np, faiss, torch
from pathlib import Path
from tqdm import tqdm
from torch.utils.data import DataLoader
from sentence_transformers import SentenceTransformer, CrossEncoder, InputExample
from sentence_transformers.cross_encoder.evaluation import CEBinaryClassificationEvaluator

ROOT          = Path(".")
DATA_DIR      = ROOT / "data"
EVAL_DIR      = ROOT / "outputs" / "eval"
TMP_DIR       = ROOT / "outputs" / "tmp"
MDL_DIR       = ROOT / "outputs" / "models"

TRAIN_NEG     = DATA_DIR / "train_with_neg.jsonl"
DEV_FILE      = DATA_DIR / "dev.jsonl"
EVAL_QA_FILE  = EVAL_DIR / "eval_qa.jsonl"
FT_BI_PATH    = MDL_DIR  / "legal_hf_finetuned" / "final"
FAISS_V4      = TMP_DIR  / "faiss_v4.index"
MAP_V4        = TMP_DIR  / "faiss_mapping_v4.jsonl"
RERANK_CSV_V5 = EVAL_DIR / "rerank_metrics_v5.csv"
GRID_CSV      = EVAL_DIR / "grid_search_skip_topk.csv"

# ── Grid search config ──
SKIP_CONFIGS  = [
    {"skip": 5,  "label": "A_skip5"},
    {"skip": 8,  "label": "B_skip8"},
    {"skip": 12, "label": "C_skip12"},
    # D (skip=14) đã biết kết quả, skip để tiết kiệm thời gian
    # {"skip": 14, "label": "D_skip14"},
]

BASE_CE_MODEL = "cross-encoder/ms-marco-MiniLM-L-6-v2"
CE_EPOCHS     = 3      # 3 epochs để grid search nhanh hơn
CE_BATCH      = 32
CE_MAX_LEN    = 256
SEED          = 42
TOP_MINE      = 30
HARD_NEG_PER  = 2
TOP_N_EVAL    = 50

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
random.seed(SEED)

print(f"Device: {DEVICE} | Grid configs: {len(SKIP_CONFIGS)} | Epochs/config: {CE_EPOCHS}")
for c in SKIP_CONFIGS:
    print(f"  {c['label']}: mine rank {c['skip']+1}-{TOP_MINE}")
print(f"  D_skip14 (baseline): đã có → R@1=0.5418, R@5=0.7245, MRR=0.6307")

## Cell 1 — Utilities

In [ ]:
def load_jsonl(path):
    rows, errors = [], 0
    with open(path, encoding="utf-8", errors="replace") as f:
        for line in f:
            line = line.strip()
            if not line: continue
            try: rows.append(json.loads(line))
            except: errors += 1
    if errors: print(f"  ⚠ {errors} errors")
    return rows

def is_hit(faiss_id, ec, mapping):
    row = mapping[faiss_id]
    for e in ec:
        ci = e.get("chunk_index", -2)
        if ci != -1 and row["chunk_index"] == ci: return True
        if (row["van_ban"]==e.get("van_ban","") and
            row["dieu"]   ==e.get("dieu",   "") and
            row["khoan"]  ==e.get("khoan",  "")): return True
    return False

def is_positive_meta(cand, meta):
    if (cand["van_ban"]==meta.get("van_ban","") and cand["van_ban"]!="" and
        cand["dieu"]  ==meta.get("dieu",   "") and
        cand["khoan"] ==meta.get("khoan",  "")): return True
    ci = meta.get("chunk_index",-2)
    return ci!=-1 and cand["chunk_index"]==ci

def avg(lst): return round(sum(lst)/len(lst), 4) if lst else 0.0

print("Utilities ✓")

## Cell 2 — Load v4 Bi-Encoder + FAISS + Pre-mine top-30
> Mine 1 lần rồi dùng cho tất cả configs — tiết kiệm thời gian.

In [ ]:
print("Loading v4 bi-encoder + FAISS...")
ft_bi      = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
index_v4   = faiss.read_index(str(FAISS_V4))
mapping_v4 = load_jsonl(MAP_V4)
print(f"  Loaded ✓ | {index_v4.ntotal} vectors")

# Load train data
train_rows = load_jsonl(TRAIN_NEG)
pos_rows   = [r for r in train_rows if r.get("label")==1]
random.seed(SEED); random.shuffle(pos_rows)
print(f"  Positive rows: {len(pos_rows)}")

# Pre-mine: retrieve top-30 cho mỗi query, lưu lại để dùng nhiều lần
print(f"Pre-mining top-{TOP_MINE} for all queries...")
premined = []   # list of dict: {query, pos_passage, meta, top30_ids}

for r in tqdm(pos_rows, desc="Pre-mining"):
    query = r.get("query","").strip()
    pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    premined.append({"query": query, "pos": pos_p, "meta": meta,
                     "ids": ids[0].tolist()})

# Pre-mine dev set
dev_rows = load_jsonl(DEV_FILE)
random.seed(SEED); random.shuffle(dev_rows)
dev_premined = []
for r in tqdm(dev_rows[:500], desc="Dev pre-mining"):
    query = r.get("query","").strip()
    pos_p = r.get("passage","").strip()
    meta  = r.get("meta",{})
    if not query or not pos_p: continue
    q_emb = ft_bi.encode([query], normalize_embeddings=True,
                          convert_to_numpy=True).astype("float32")
    _, ids = index_v4.search(q_emb, TOP_MINE)
    dev_premined.append({"query": query, "pos": pos_p, "meta": meta,
                         "ids": ids[0].tolist()})

print(f"Pre-mining done: {len(premined)} train, {len(dev_premined)} dev")

# Giải phóng bi-encoder để nhường VRAM cho CE training
del ft_bi; gc.collect()
torch.cuda.empty_cache() if DEVICE == "cuda" else None
print("VRAM cleared ✓")

## Cell 3 — Grid Search Loop
> Với mỗi SKIP_TOP_K: tạo training set → train CE → evaluate → ghi kết quả

In [ ]:
eval_qa    = load_jsonl(EVAL_QA_FILE)
grid_results = []

# Kết quả của D_skip14 đã biết
grid_results.append({
    "label":"D_skip14","skip":14,
    "R@1":0.5418,"R@3":0.6873,"R@5":0.7245,"MRR@10":0.6307,
    "train_size":"N/A","neg_per_q":"~1.0"
})

def build_training_set(premined_data, skip_k, hard_neg_per):
    """Tạo training set với mining range [skip_k:TOP_MINE]."""
    examples = []
    neg_counts = []
    for item in premined_data:
        examples.append(InputExample(texts=[item["query"],item["pos"]], label=1.0))
        pool = item["ids"][skip_k:]    # ← KEY: dùng range [skip_k:]
        added = 0
        for fid in pool:
            if fid < 0 or added >= hard_neg_per: break
            cand = mapping_v4[fid]
            if cand["passage"] == item["pos"]: continue
            if is_positive_meta(cand, item["meta"]): continue
            examples.append(InputExample(texts=[item["query"],cand["passage"]], label=0.0))
            added += 1
        neg_counts.append(added)
    return examples, neg_counts

def build_dev_set(dev_premined_data, skip_k):
    dev_ex = []
    for item in dev_premined_data:
        dev_ex.append(InputExample(texts=[item["query"],item["pos"]], label=1.0))
        for fid in item["ids"][skip_k:]:
            if fid < 0: break
            cand = mapping_v4[fid]
            if cand["passage"]==item["pos"]: continue
            if is_positive_meta(cand, item["meta"]): continue
            dev_ex.append(InputExample(texts=[item["query"],cand["passage"]], label=0.0))
            break
    return dev_ex

for cfg in SKIP_CONFIGS:
    skip_k = cfg["skip"]; label = cfg["label"]
    print(f"\n{'='*60}")
    print(f"Config {label}: SKIP_TOP_K={skip_k}, range=[{skip_k+1}:{TOP_MINE}]")
    print(f"{'='*60}")

    # Build training data
    train_ex, neg_counts = build_training_set(premined, skip_k, HARD_NEG_PER)
    dev_ex   = build_dev_set(dev_premined, skip_k)
    avg_neg  = round(sum(neg_counts)/max(len(neg_counts),1), 2)
    pos_ratio= round(sum(1 for e in train_ex if e.label==1)/len(train_ex), 2)
    print(f"  Train: {len(train_ex)} | Dev: {len(dev_ex)} | avg_neg/q={avg_neg} | pos_ratio={pos_ratio:.0%}")

    # Train CE
    ce = CrossEncoder(BASE_CE_MODEL, num_labels=1, max_length=CE_MAX_LEN, device=DEVICE)
    evaluator = CEBinaryClassificationEvaluator.from_input_examples(dev_ex, name=label)
    warmup = int(len(train_ex)/CE_BATCH * CE_EPOCHS * 0.1)

    t0 = time.time()
    ce_dir = MDL_DIR / f"ce_grid_{label}"
    ce_dir.mkdir(parents=True, exist_ok=True)
    random.seed(SEED)
    ce.fit(
        train_dataloader=DataLoader(train_ex, shuffle=True, batch_size=CE_BATCH),
        evaluator=evaluator, epochs=CE_EPOCHS,
        warmup_steps=warmup, output_path=str(ce_dir),
        use_amp=(DEVICE=="cuda"),
    )
    print(f"  Training done in {round((time.time()-t0)/60,1)} min")

    # Evaluate
    # Reload bi-encoder cho evaluation
    ft_bi_eval = SentenceTransformer(str(FT_BI_PATH), device=DEVICE)
    r_r = {"R@1":[],"R@3":[],"R@5":[],"MRR@10":[]}

    for item in tqdm(eval_qa, desc=f"Eval {label}", leave=False):
        query = item["query"]; ec = item["expected_citations"]
        q_emb = ft_bi_eval.encode([query], normalize_embeddings=True,
                                   convert_to_numpy=True).astype("float32")
        _, ids = index_v4.search(q_emb, TOP_N_EVAL)
        ids    = ids[0].tolist()
        cands  = [(mapping_v4[i]["passage"],i) for i in ids if i>=0]
        rscore = ce.predict([[query,c[0]] for c in cands], batch_size=32) if cands else []
        ranked = sorted(zip(rscore,[c[1] for c in cands]),reverse=True)
        r_ids  = [r[1] for r in ranked]
        for k,key in [(1,"R@1"),(3,"R@3"),(5,"R@5")]:
            r_r[key].append(1 if any(is_hit(i,ec,mapping_v4) for i in r_ids[:k]) else 0)
        mrr = 0.0
        for rank,i in enumerate(r_ids[:10],1):
            if is_hit(i,ec,mapping_v4): mrr=1.0/rank; break
        r_r["MRR@10"].append(mrr)

    del ft_bi_eval; gc.collect()
    torch.cuda.empty_cache() if DEVICE=="cuda" else None

    res = {"label":label,"skip":skip_k,
           "R@1":avg(r_r["R@1"]),"R@3":avg(r_r["R@3"]),
           "R@5":avg(r_r["R@5"]),"MRR@10":avg(r_r["MRR@10"]),
           "train_size":len(train_ex),"neg_per_q":avg_neg}
    grid_results.append(res)
    print(f"  → R@1={res['R@1']}, R@3={res['R@3']}, R@5={res['R@5']}, MRR={res['MRR@10']}")

print("\n✓ Grid search complete!")

## Cell 4 — Tổng hợp kết quả & Lưu

In [ ]:
# Sort by R@1 desc
grid_results_sorted = sorted(grid_results, key=lambda x: x["R@1"], reverse=True)

print("\n" + "="*88)
print(f"  {'Config':<12} {'Skip':>6} {'R@1':>8} {'R@3':>8} {'R@5':>8} {'MRR@10':>8} {'Train':>8}")
print("="*88)
for r in grid_results_sorted:
    marker = " ★" if r==grid_results_sorted[0] else ""
    print(f"  {r['label']:<12} {r['skip']:>6} {r['R@1']:>8.4f} {r['R@3']:>8.4f} {r['R@5']:>8.4f} {r['MRR@10']:>8.4f} {str(r['train_size']):>8}{marker}")
print("="*88)

best = grid_results_sorted[0]
print(f"\n★ Best config: {best['label']} (SKIP_TOP_K={best['skip']})")
print(f"  R@1={best['R@1']}, R@3={best['R@3']}, R@5={best['R@5']}, MRR={best['MRR@10']}")

# Lưu CSV
with open(GRID_CSV, "w", newline="", encoding="utf-8") as f:
    w = csv.DictWriter(f, fieldnames=["label","skip","R@1","R@3","R@5","MRR@10","train_size","neg_per_q"])
    w.writeheader(); w.writerows(grid_results_sorted)
print(f"Saved → {GRID_CSV} ✓")